In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings


# -------------------------
# Sample Documents
# -------------------------
documents = [
    "Subscription pricing works best for SaaS.",
    "Freemium model failed in enterprise.",
    "Tiered pricing increased revenue.",
    "Customers like flexible pricing."
]


# -------------------------
# Vector Store (ISSUE #1)
# -------------------------
embeddings = OpenAIEmbeddings()

vector_store = Chroma.from_texts(
    documents,
    embedding=embeddings
)


# -------------------------
# Retrieval (ISSUE #2)
# -------------------------
def retrieve(query):

    # ❌ wrong k (too high)
    return vector_store.similarity_search(query, k=10)


# -------------------------
# Ranking (ISSUE #3)
# -------------------------
def rank(docs):

    # ❌ no ranking logic
    return docs


# -------------------------
# Context Builder (ISSUE #4)
# -------------------------
def build_context(docs):

    # ❌ only takes first doc (loses context)
    return docs[0].page_content


# -------------------------
# LLM Call (ISSUE #5)
# -------------------------
def generate_answer(query):

    docs = retrieve(query)

    docs = rank(docs)

    context = build_context(docs)

    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    messages = [
        SystemMessage(content="You are helpful."),
        HumanMessage(content=f"{query}")  # ❌ context not used
    ]

    return llm.invoke(messages).content


# -------------------------
# Run
# -------------------------
print(generate_answer("What pricing strategy should we use?"))

In [ ]:
def retrieve(query):
    return vector_store.similarity_search(query, k=3)

In [ ]:
def rank(docs, query):

    scored = []

    for doc in docs:
        score = 0
        for word in query.lower().split():
            if word in doc.page_content.lower():
                score += 1

        scored.append((doc, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    return [d for d, _ in scored]

In [ ]:
def build_context(docs):

    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
messages = [
    SystemMessage(content="You are a product strategist."),
    HumanMessage(content=f"""
Context:
{context}

Question:
{query}
""")
]

In [ ]:
# cache results
cache = {}